# Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved.
# SPDX-License-Identifier: MIT-0

# Automated Blood Sample Quality Control — Pipeline Validation Walkthrough

> **Illustrative use only.** This notebook treats platelet presence as a stand-in "defect" class purely to demonstrate the pipeline on a real public dataset. It is not a clinical or diagnostic application, and the labels do not reflect actual blood-sample quality.

This notebook serves two purposes:

1. **Validation walkthrough** — run it start-to-finish against a deployed stack. Every cell executing without error confirms the end-to-end path (pipeline execution, registry, inference). The notebook ships without saved outputs; execute it in your own account to produce them.
2. **Usage guide** — with code cells collapsed, this reads as a clean research report showing how to execute the pipeline, retrieve models, and run inference.

**Dataset:** [BCCD](https://github.com/Shenggan/BCCD_Dataset) (Blood Cell Count and Detection) — 364 blood smear microscopy images, MIT license.
**Classes:** RBC (red blood cells), WBC (white blood cells), Platelets.
**Models:**
- **Anomaly detector** (convolutional autoencoder) — trained on normal slides (RBC+WBC only), flags anomalous slides via reconstruction error
- **Classifier** (ResNet18) — classifies slides as good vs defect (Platelets present)

Both models are evaluated against quality gates (anomaly AUROC >= 0.8, classifier balanced accuracy >= 0.6 AND defect recall >= 0.5) and registered in the SageMaker Model Registry with ONNX artifacts.

---

## 0 — Prerequisites: Download the BCCD Dataset

The inference sections below use real BCCD test images from a local download.
This step clones the dataset, converts it to the anomalib layout, and uploads
tarballs to the pipeline S3 bucket so the pipeline execution in Section 1 can
train on real data. Run it once before executing the rest of the notebook.

In [ ]:
!python ../scripts/download_bccd.py --output-dir ../data/bccd --upload-s3

In [ ]:
import json
import shutil
import tarfile
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path

import boto3
import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
from IPython.display import Markdown, display
from PIL import Image

REGION = boto3.session.Session().region_name
if not REGION:
    raise RuntimeError(
        "AWS region could not be resolved from the boto3 session. "
        "Configure AWS_REGION/AWS_DEFAULT_REGION or a profile with a default region."
    )
sm = boto3.client("sagemaker", region_name=REGION)
s3 = boto3.client("s3", region_name=REGION)
cfn = boto3.client("cloudformation", region_name=REGION)


def get_stack_outputs(stack_name):
    resp = cfn.describe_stacks(StackName=stack_name)
    return {o["OutputKey"]: o["OutputValue"] for o in resp["Stacks"][0]["Outputs"]}


outputs = get_stack_outputs("SageMakerCVPipelineStack")
PIPELINE_NAME = outputs["CVPipelineName"]
BUCKET = outputs["CVBucketName"]

print(f"Pipeline:  {PIPELINE_NAME}")
print(f"S3 Bucket: {BUCKET}")
print(f"Region:    {REGION}")
print(f"Timestamp: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}")

## 1 — Pipeline Execution

Start a pipeline execution trained on **real BCCD blood cell images** uploaded to S3.
The `TrainingDataS3Uri` parameter tells the preprocessing step to download real data instead of generating synthetic images.

The pipeline runs 13 steps:
preprocess → fetch latest model (x2) → train (x2) → evaluate (x2) → quality gate (x2) → Neo compile (x2) → register (x2).

Measured runtime: ~15 minutes end-to-end on `ml.m5.xlarge` with synthetic data (Jul 2026, us-east-2), including both Neo compilations; real BCCD data adds download/upload time.

In [ ]:
TRAINING_DATA_S3_URI = f"s3://{BUCKET}/data"

resp = sm.start_pipeline_execution(
    PipelineName=PIPELINE_NAME,
    PipelineParameters=[
        {"Name": "TrainingInstanceType", "Value": "ml.m5.xlarge"},
        {"Name": "NumSamples", "Value": "400"},
        {"Name": "TrainingDataS3Uri", "Value": TRAINING_DATA_S3_URI},
    ],
)
EXECUTION_ARN = resp["PipelineExecutionArn"]
print(f"Started: {EXECUTION_ARN}")
print(f"Training data: {TRAINING_DATA_S3_URI}")
print(f"Polling every 60s...\n")

while True:
    desc = sm.describe_pipeline_execution(PipelineExecutionArn=EXECUTION_ARN)
    status = desc["PipelineExecutionStatus"]
    elapsed = ""
    if "CreationTime" in desc:
        dt = (datetime.now(timezone.utc) - desc["CreationTime"].replace(tzinfo=timezone.utc)).total_seconds()
        elapsed = f" ({int(dt)}s elapsed)"
    print(f"  {time.strftime('%H:%M:%S')} — {status}{elapsed}")
    if status not in ("Executing", "Stopping"):
        break
    time.sleep(60)

print(f"\nPipeline finished: {status}")
assert status == "Succeeded", f"Pipeline failed with status: {status}"

### Pipeline Step Results

Verify all 13 steps completed successfully.

In [ ]:
steps = sm.list_pipeline_execution_steps(PipelineExecutionArn=EXECUTION_ARN)["PipelineExecutionSteps"]
steps.sort(key=lambda s: s.get("StartTime", datetime.min.replace(tzinfo=timezone.utc)))

rows = []
for step in steps:
    name = step["StepName"]
    status = step["StepStatus"]
    start = step.get("StartTime")
    end = step.get("EndTime")
    duration = f"{(end - start).total_seconds():.0f}s" if start and end else "—"
    rows.append(f"| {name} | {status} | {duration} |")

table = "| Step | Status | Duration |\n|---|---|---|\n" + "\n".join(rows)
display(Markdown(table))

## 2 — Model Registry

The pipeline registers both models in the SageMaker Model Registry with quality metrics attached.
Each model version includes the ONNX artifact and the evaluation report that passed the quality gate.

In [ ]:
MODEL_GROUPS = ["sm1k-anomaly-model-group", "sm1k-classifier-model-group"]


# In a fleet deployment every camera registers into the same group, so
# filter on the CameraId lineage metadata; taking the newest package
# unfiltered could hand you another camera's model.
CAMERA_ID = "default"  # match the CameraId parameter of your pipeline run


def get_latest_model(group_name, camera_id=CAMERA_ID):
    paginator = sm.get_paginator("list_model_packages")
    pages = paginator.paginate(
        ModelPackageGroupName=group_name,
        SortBy="CreationTime", SortOrder="Descending",
    )
    for page in pages:
        for summary in page["ModelPackageSummaryList"]:
            desc = sm.describe_model_package(ModelPackageName=summary["ModelPackageArn"])
            if desc.get("CustomerMetadataProperties", {}).get("CameraId") == camera_id:
                return desc
    raise RuntimeError(f"No package in {group_name} with CameraId={camera_id}")


def fetch_metrics(desc):
    uri = desc.get("ModelMetrics", {}).get("ModelQuality", {}).get("Statistics", {}).get("S3Uri")
    if not uri:
        return {}
    bucket = uri.split("/")[2]
    key = "/".join(uri.split("/")[3:])
    return json.loads(s3.get_object(Bucket=bucket, Key=key)["Body"].read())


anomalib_desc = get_latest_model("sm1k-anomaly-model-group")
classifier_desc = get_latest_model("sm1k-classifier-model-group")
anomalib_metrics = fetch_metrics(anomalib_desc)
classifier_metrics = fetch_metrics(classifier_desc)

rows = []
rows.append("| Model | Version | Approval | Key Metric | Value | Gate |")
rows.append("|---|---|---|---|---|---|")

av = anomalib_desc["ModelPackageArn"].split("/")[-1]
auroc = anomalib_metrics.get("binary_classification_metrics", {}).get("auc", {}).get("value", "—")
auroc_pass = "pass (>= 0.8)" if isinstance(auroc, (int, float)) and auroc >= 0.8 else "fail"
rows.append(f"| Anomaly detector | v{av} | {anomalib_desc['ModelApprovalStatus']} | AUROC | {auroc} | {auroc_pass} |")

cv = classifier_desc["ModelPackageArn"].split("/")[-1]
balanced_accuracy = classifier_metrics.get("balanced_accuracy", "—")
defect_recall = classifier_metrics.get("defect_recall", "—")
bal_pass = "pass (>= 0.6)" if isinstance(balanced_accuracy, (int, float)) and balanced_accuracy >= 0.6 else "fail"
rows.append(f"| Classifier | v{cv} | {classifier_desc['ModelApprovalStatus']} | Balanced accuracy | {balanced_accuracy} | {bal_pass} |")
recall_pass = "pass (>= 0.5)" if isinstance(defect_recall, (int, float)) and defect_recall >= 0.5 else "fail"
rows.append(f"| Classifier | v{cv} | {classifier_desc['ModelApprovalStatus']} | Defect recall | {defect_recall} | {recall_pass} |")

display(Markdown("\n".join(rows)))

## 3 — Download ONNX Models

Each registered model stores a `model.tar.gz` containing the exported ONNX artifact.
We download both models for local inference.

In [ ]:
TMP = Path(tempfile.mkdtemp(prefix="sm1k-report-"))


def download_onnx(desc, dest_dir):
    uri = desc["InferenceSpecification"]["Containers"][0]["ModelDataUrl"]
    bucket = uri.split("/")[2]
    key = "/".join(uri.split("/")[3:])
    dest = Path(dest_dir)
    dest.mkdir(parents=True, exist_ok=True)
    tar_path = dest / "model.tar.gz"
    s3.download_file(bucket, key, str(tar_path))
    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall(dest, filter="data")
    onnx_path = next(dest.rglob("*.onnx"))
    size_kb = onnx_path.stat().st_size / 1024
    return onnx_path, size_kb


anomalib_onnx, anomalib_size = download_onnx(anomalib_desc, TMP / "anomalib")
classifier_onnx, classifier_size = download_onnx(classifier_desc, TMP / "classifier")

display(Markdown(
    f"| Model | File | Size |\n|---|---|---|\n"
    f"| Anomaly detector | `{anomalib_onnx.name}` | {anomalib_size:.0f} KB |\n"
    f"| Classifier | `{classifier_onnx.name}` | {classifier_size:.0f} KB |"
))

## 4 — Anomaly Detection Inference

The anomaly model is a convolutional autoencoder trained on "normal" blood smear slides
(containing only RBC and WBC). It learns to reconstruct normal slides — high reconstruction
error signals an anomalous sample (e.g., unexpected Platelets).

We test with real BCCD images: one normal slide and one containing Platelets.

In [ ]:
IMAGE_SIZE = 256
sess = ort.InferenceSession(str(anomalib_onnx), providers=["CPUExecutionProvider"])
inp_meta = sess.get_inputs()[0]
out_meta = sess.get_outputs()[0]


def anomaly_score(img):
    arr = np.array(img.resize((IMAGE_SIZE, IMAGE_SIZE)), dtype=np.float32).transpose(2, 0, 1) / 255.0
    batch = arr[np.newaxis, ...]
    recon = sess.run([out_meta.name], {inp_meta.name: batch})[0]
    return float(np.mean((recon - batch) ** 2))


# Load real BCCD test images from the local download (Section 0)
bccd_test_anomalib = Path("../data/bccd/anomalib/test")
good_images = sorted((bccd_test_anomalib / "good").glob("*.png"))
defect_images = sorted((bccd_test_anomalib / "defect").glob("*.png"))

# Pick representative samples
normal_img = Image.open(good_images[0]).convert("RGB")
anomalous_img = Image.open(defect_images[0]).convert("RGB")

score_normal = anomaly_score(normal_img)
score_anomalous = anomaly_score(anomalous_img)
ratio = score_anomalous / score_normal if score_normal > 0 else float("inf")

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(normal_img)
axes[0].set_title(f"Normal (RBC + WBC only)\nerror = {score_normal:.6f}", fontsize=10)
axes[0].axis("off")
axes[1].imshow(anomalous_img)
axes[1].set_title(f"Anomalous (Platelets present)\nerror = {score_anomalous:.6f}", fontsize=10)
axes[1].axis("off")
fig.suptitle(f"Anomaly Detection on BCCD Blood Smear Images — {ratio:.1f}x error difference", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Normal score:    {score_normal:.6f}  ({good_images[0].name})")
print(f"Anomalous score: {score_anomalous:.6f}  ({defect_images[0].name})")
print(f"Ratio:           {ratio:.1f}x")
print(f"\nTest set: {len(good_images)} normal, {len(defect_images)} anomalous slides")

## 5 — Classifier Inference

The ResNet18 classifier labels slides as **good** (RBC+WBC only) or **defect** (Platelets present).
We run ONNX inference on one image from each class and compare the predictions.

In [ ]:
CLASSIFIER_IMAGE_SIZE = 224
CLASS_NAMES = ["good", "defect"]

clf_sess = ort.InferenceSession(str(classifier_onnx), providers=["CPUExecutionProvider"])
clf_inp = clf_sess.get_inputs()[0]
clf_out = clf_sess.get_outputs()[0]


def classify(img):
    arr = np.array(img.resize((CLASSIFIER_IMAGE_SIZE, CLASSIFIER_IMAGE_SIZE)), dtype=np.float32).transpose(2, 0, 1) / 255.0
    batch = arr[np.newaxis, ...]
    logits = clf_sess.run([clf_out.name], {clf_inp.name: batch})[0][0]
    probs = np.exp(logits - np.max(logits))
    probs /= probs.sum()
    pred = int(np.argmax(probs))
    return CLASS_NAMES[pred], float(probs[pred])


pred_good, conf_good = classify(normal_img)
pred_defect, conf_defect = classify(anomalous_img)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(normal_img)
axes[0].set_title(f"True: good\nPredicted: {pred_good} ({conf_good:.2f})", fontsize=10)
axes[0].axis("off")
axes[1].imshow(anomalous_img)
axes[1].set_title(f"True: defect\nPredicted: {pred_defect} ({conf_defect:.2f})", fontsize=10)
axes[1].axis("off")
fig.suptitle("ResNet18 Defect Classification on BCCD Blood Smear Images", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Good slide   -> predicted {pred_good} (confidence {conf_good:.4f})")
print(f"Defect slide -> predicted {pred_defect} (confidence {conf_defect:.4f})")

## 6 — Summary

Final validation summary. If this cell renders, all checks passed.

In [ ]:
summary = f"""| Check | Result |
|---|---|
| Pipeline execution | {status} |
| All steps completed | {len(steps)} steps |
| Anomaly detector registered | v{av}, AUROC={auroc} |
| Classifier registered | v{cv}, balanced_accuracy={balanced_accuracy}, defect_recall={defect_recall} |
| Anomaly ONNX inference | {ratio:.1f}x anomaly separation |
| Classifier ONNX inference | good -> {pred_good}, defect -> {pred_defect} |
| Report generated | {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')} |"""

display(Markdown(summary))

# Cleanup
shutil.rmtree(TMP, ignore_errors=True)
print(f"\nTemporary files cleaned up. Report complete.")